# Soybean Rust Early Warning — Phase 2, Notebook 2 (v2): Rust-risk map for one season

**v2:** uses the 14 features of main pipeline v6 (no wind direction) and masks lakes (e.g. Lake Malawi, Bangweulu, Mweru, Kariba).

This notebook turns the trained trial model into a **map of rust-outbreak risk** across Malawi and Zambia for one rainy season.

| Step | What happens |
|---|---|
| 1 | Train the outbreak and severity models on the 78 assessed trials, **leaving out the map season** so the map is a genuine prediction |
| 2 | Build a grid of cells covering Malawi and Zambia |
| 3 | Sowing date per cell: **later of 21 December or 7 days after season onset** (onset rule C from CHIRPS; tested in Notebook 1) |
| 4 | R4 date per cell: sowing + 69 days (median of the trials) |
| 5 | Hourly ECMWF IFS weather for the 14 days before R4 in every cell, and the same 16 features |
| 6 | Predict outbreak probability, alert level and severity per cell; flag cells whose weather lies outside the trials' range |
| 7 | Compare the map with what the season's trials actually recorded |
| 8 | Save maps as PNG, GeoTIFF (for QGIS/ArcGIS) and CSV |

**Needed files:** `v6_trials_features.csv` from the main pipeline (upload when asked; the v5 file also works).
**Run time:** about 10–20 minutes the first time at the default 0.2° grid; results are cached.

## 0. Setup and settings

In [ ]:
import os, io, json, time, warnings, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import requests
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); plt.rcParams['figure.dpi'] = 110
for pkg in ['rasterio', 'shapely']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import rasterio
from rasterio.windows import from_bounds
from rasterio.transform import from_origin
from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely.prepared import prep

# ---------------- SETTINGS ----------------
TRIALS_CSV   = 'v6_trials_features.csv'   # saved by the main pipeline (Section 12); v5 file also works
MAP_SEASON   = 2023          # season year: 2023 = 2023/24 (dry, low-rust season); 2020 = 2020/21 (wet, high-rust)
GRID_DEG     = 0.2           # grid spacing in degrees (0.2 ≈ 22 km; 0.1 ≈ 11 km, about 4x more cells and download)
COUNTRIES    = ['Malawi', 'Zambia']
SOW_FIXED    = (12, 21)      # typical sowing date: 21 December
ONSET_LAG    = 7             # ... but never earlier than onset + 7 days
ONSET_MM, ONSET_DAYS = 25, 3 # onset rule C: 25 mm over 3 days, from 1 November
R4_DAYS      = 69            # median days from sowing to R4 in the trials
WINDOW_DAYS  = 14
LEAVE_OUT_MAP_SEASON = True  # train without the map season, so the map can be checked against its trials
CACHE_DIR    = 'weather_cache'
T_LOW, T_HIGH, RH_WET, MIN_WET_HOURS = 17, 29, 90, 6
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
if not os.path.exists(TRIALS_CSV):
    try:
        from google.colab import files
        print('Please upload', TRIALS_CSV); up = files.upload(); TRIALS_CSV = list(up.keys())[0]
    except ImportError:
        raise FileNotFoundError(f'{TRIALS_CSV} not found.')
print('Using', TRIALS_CSV)

## 1. Train the models on the trials

The same models as the main pipeline: **logistic regression** for outbreak probability and **random forest** for severity, with the same 16 weather features.
With `LEAVE_OUT_MAP_SEASON = True`, the map season's trials are removed from training, so the map is an out-of-sample prediction.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

FEATURES = ['wet_hours', 'wet_days6', 'infect_days', 'tmin', 'tmax', 'hot_days', 'hours_T17_29', 'rain_total',
            'rain_days', 'rh_mean', 'vpd_mean', 'srad', 'wind', 'soilm']      # 14 features (no wind direction, as in v6)
trials = pd.read_csv(TRIALS_CSV, parse_dates=['sowing'])
train = trials[trials.year != MAP_SEASON] if LEAVE_OUT_MAP_SEASON else trials
check = trials[(trials.year == MAP_SEASON) & (trials.season == 2)]    # rainy-season trials only

outbreak_model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                               LogisticRegression(C=0.3, class_weight='balanced', max_iter=2000)).fit(train[FEATURES], train.outbreak)
severity_model = make_pipeline(SimpleImputer(strategy='median'),
                               RandomForestRegressor(n_estimators=500, min_samples_leaf=4, max_features=0.5,
                                                     random_state=42, n_jobs=-1)).fit(train[FEATURES], train.severity)
print(f'Trained on {len(train)} trials ({train.outbreak.sum()} outbreaks); '
      f'{len(check)} rainy-season trials of {MAP_SEASON}/{str(MAP_SEASON + 1)[-2:]} kept for checking the map')

## 2. Grid over Malawi and Zambia

Country outlines and lakes come from Natural Earth (public domain). Cell centres inside either country are kept, **except those on a lake**.

In [ ]:
NE_URL = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson'
ne_path = os.path.join(CACHE_DIR, 'ne_50m_countries.geojson')
if not os.path.exists(ne_path):
    open(ne_path, 'wb').write(requests.get(NE_URL, timeout=120).content)
ne = json.load(open(ne_path))
shapes = {f['properties'].get('ADMIN', f['properties'].get('NAME')): shape(f['geometry']) for f in ne['features']}
LAKES_URL = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_lakes.geojson'
lk_path = os.path.join(CACHE_DIR, 'ne_10m_lakes.geojson')
if not os.path.exists(lk_path):
    open(lk_path, 'wb').write(requests.get(LAKES_URL, timeout=180).content)
lakes_all = [shape(f['geometry']) for f in json.load(open(lk_path))['features']]
study = unary_union([shapes[c] for c in COUNTRIES]); study_p = prep(study)
lakes = unary_union([l for l in lakes_all if l.intersects(study)]); lakes_p = prep(lakes)
minx, miny, maxx, maxy = study.bounds

lons = np.round(np.arange(np.floor(minx / GRID_DEG) * GRID_DEG + GRID_DEG / 2, maxx, GRID_DEG), 4)
lats = np.round(np.arange(np.ceil(maxy / GRID_DEG) * GRID_DEG - GRID_DEG / 2, miny, -GRID_DEG), 4)   # north to south
grid = pd.DataFrame([(i, j, la, lo) for i, la in enumerate(lats) for j, lo in enumerate(lons)],
                    columns=['row', 'col', 'lat', 'lon'])
inside = np.array([study_p.contains(Point(lo, la)) for la, lo in zip(grid.lat, grid.lon)])
on_lake = np.array([lakes_p.contains(Point(lo, la)) for la, lo in zip(grid.lat, grid.lon)])
print(f'Cells on lakes removed: {(inside & on_lake).sum()}')
grid = grid[inside & ~on_lake].reset_index(drop=True)
grid['country'] = ['Malawi' if shapes['Malawi'].contains(Point(lo, la)) else 'Zambia' for la, lo in zip(grid.lat, grid.lon)]
grid['cell'] = [f'c{r}_{c}' for r, c in zip(grid.row, grid.col)]
print(f'{len(grid)} cells of {GRID_DEG}° (', grid.country.value_counts().to_dict(), ')')

## 3. Season onset and sowing date per cell (CHIRPS)

Daily CHIRPS rainfall is read for the whole study area (1 October to 31 March). Onset is the first day from 1 November with at least
25 mm over 3 days (rule C). Sowing = the **later** of 21 December or onset + 7 days.

In [ ]:
CHC_URL = 'https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/cogs/p05/{d:%Y}/chirps-v2.0.{d:%Y.%m.%d}.cog'
cdir = os.path.join(CACHE_DIR, f'chirps_area_{MAP_SEASON}'); os.makedirs(cdir, exist_ok=True)
GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.cog,.tif',
                GDAL_HTTP_MAX_RETRY='4', GDAL_HTTP_RETRY_DELAY='3')
B = (minx - 0.1, miny - 0.1, maxx + 0.1, maxy + 0.1)

def chirps_area(day):
    path = os.path.join(cdir, f'{day:%Y%m%d}.npz')
    if os.path.exists(path):
        z = np.load(path); return z['a'], tuple(z['t'])
    for attempt in range(3):
        try:
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(CHC_URL.format(d=day)) as src:
                    w = from_bounds(*B, transform=src.transform)
                    a = src.read(1, window=w).astype('float32'); t = src.window_transform(w)
                    a[(a < 0) | (a == src.nodata)] = np.nan
            np.savez_compressed(path, a=a, t=np.array(t)[:6]); return a, tuple(np.array(t)[:6])
        except Exception:
            time.sleep(3)
    return None, None

days = pd.date_range(f'{MAP_SEASON}-10-01', f'{MAP_SEASON + 1}-03-31')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    got = list(pool.map(chirps_area, days))
ok = [(d, a, t) for d, (a, t) in zip(days, got) if a is not None]
print(f'CHIRPS: {len(ok)} of {len(days)} days read in {time.time() - t0:.0f} s')

a0, t = ok[0][1], ok[0][2]          # affine: (a, b, c, d, e, f) -> x = c + col*a, y = f + row*e
px_col = ((grid.lon - t[2]) / t[0]).astype(int).clip(0, a0.shape[1] - 1)
px_row = ((grid.lat - t[5]) / t[4]).astype(int).clip(0, a0.shape[0] - 1)
rain_cells = pd.DataFrame({d: a[px_row, px_col] for d, a, _ in ok}).T      # rows = days, columns = cells
rain_cells.columns = grid.cell

def onset_rule_c(r):
    wet = r.fillna(0).rolling(ONSET_DAYS).sum().shift(-(ONSET_DAYS - 1))
    wet = wet[(wet.index >= pd.Timestamp(MAP_SEASON, 11, 1)) & (wet.index <= pd.Timestamp(MAP_SEASON + 1, 2, 28))]
    hit = wet[wet >= ONSET_MM]
    return hit.index[0] if len(hit) else pd.NaT

grid['onset'] = [onset_rule_c(rain_cells[c]) for c in grid.cell]
fixed = pd.Timestamp(MAP_SEASON, *SOW_FIXED)
grid['sowing'] = [max(fixed, o + pd.Timedelta(days=ONSET_LAG)) if pd.notna(o) else pd.NaT for o in grid.onset]
grid['R4_date'] = grid.sowing + pd.Timedelta(days=R4_DAYS)
grid['win_start'] = grid.R4_date - pd.Timedelta(days=WINDOW_DAYS)
print(f'Onset found in {grid.onset.notna().mean():.0%} of cells | median onset {grid.onset.median():%d %b} | '
      f'sowing = 21 Dec in {(grid.sowing == fixed).mean():.0%} of cells, later elsewhere (latest {grid.sowing.max():%d %b})')

## 4. Hourly IFS weather for each cell's 14-day window

Open-Meteo accepts many locations per request. Cells with the same window are requested together in groups of 50,
with pauses to respect the free service's limits; every cell is cached, so an interrupted download can simply be re-run.

In [ ]:
OM_URL = 'https://archive-api.open-meteo.com/v1/archive'
HOURLY = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'shortwave_radiation',
          'wind_speed_10m', 'wind_direction_10m', 'vapour_pressure_deficit', 'soil_moisture_0_to_7cm']
gdir = os.path.join(CACHE_DIR, f'grid_ifs_{MAP_SEASON}_{GRID_DEG}'); os.makedirs(gdir, exist_ok=True)
cpath = lambda cell, start: os.path.join(gdir, f'{cell}_{start:%Y%m%d}.csv')

def fetch_group(cells_df, start, end):
    params = dict(latitude=','.join(f'{x:.4f}' for x in cells_df.lat), longitude=','.join(f'{x:.4f}' for x in cells_df.lon),
                  models='ecmwf_ifs', start_date=f'{start:%Y-%m-%d}', end_date=f'{end:%Y-%m-%d}',
                  hourly=','.join(HOURLY), timezone='auto', wind_speed_unit='ms', cell_selection='land')
    for attempt in range(6):
        r = requests.get(OM_URL, params=params, timeout=180)
        if r.status_code == 429:
            wait = 65 * (attempt + 1); print(f'  rate limit reached, waiting {wait} s ...'); time.sleep(wait); continue
        if r.status_code == 400:
            print('  rejected:', r.json().get('reason')); return
        r.raise_for_status()
        js = r.json(); js = js if isinstance(js, list) else [js]
        for (_, c), res in zip(cells_df.iterrows(), js):
            h = pd.DataFrame(res['hourly']); h['time'] = pd.to_datetime(h['time'])
            h.to_csv(cpath(c.cell, start), index=False)
        return
    print('  gave up on this group; re-run the cell later to continue')

todo = grid.dropna(subset=['win_start'])
todo = todo[[not os.path.exists(cpath(c, s)) for c, s in zip(todo.cell, todo.win_start)]]
print(f'{len(todo)} cells to download ({len(grid) - len(todo)} already cached)')
t0 = time.time()
for start, grp in todo.groupby('win_start'):
    end = start + pd.Timedelta(days=WINDOW_DAYS - 1)
    for k in range(0, len(grp), 50):
        fetch_group(grp.iloc[k:k + 50], start, end); time.sleep(5)
print(f'Done in {(time.time() - t0) / 60:.1f} min')

## 5. Weather features per cell (same 16 as the trials)

In [ ]:
def window_features(h):
    if h is None or h.empty:
        return {}
    h = h.copy(); h['date'] = h['time'].dt.normalize(); h['wet'] = h['relative_humidity_2m'] >= RH_WET
    d = h.groupby('date').agg(tmin=('temperature_2m', 'min'), tmax=('temperature_2m', 'max'),
                              rain=('precipitation', 'sum'), wet_h=('wet', 'sum'))
    d['wet_T'] = h[h['wet']].groupby('date')['temperature_2m'].mean().reindex(d.index)
    infect = (d['wet_h'] >= MIN_WET_HOURS) & d['wet_T'].between(T_LOW, T_HIGH)
    wd = np.deg2rad(h['wind_direction_10m'])
    return {'wet_hours': h['wet'].sum(), 'wet_days6': (d['wet_h'] >= MIN_WET_HOURS).sum(), 'infect_days': infect.sum(),
            'tmin': d['tmin'].mean(), 'tmax': d['tmax'].mean(), 'hot_days': (d['tmax'] > 30).sum(),
            'hours_T17_29': h['temperature_2m'].between(T_LOW, T_HIGH).sum(), 'rain_total': d['rain'].sum(),
            'rain_days': (d['rain'] >= 1).sum(), 'rh_mean': h['relative_humidity_2m'].mean(),
            'vpd_mean': h['vapour_pressure_deficit'].mean(), 'srad': h['shortwave_radiation'].mean(),
            'wind': h['wind_speed_10m'].mean(), 'wind_sin': np.sin(wd).mean(), 'wind_cos': np.cos(wd).mean(),
            'soilm': h['soil_moisture_0_to_7cm'].mean()}

rows = []
for _, c in grid.iterrows():
    p = cpath(c.cell, c.win_start) if pd.notna(c.win_start) else None
    h = pd.read_csv(p, parse_dates=['time']) if p and os.path.exists(p) else None
    rows.append({'cell': c.cell, **window_features(h)})
grid = grid.merge(pd.DataFrame(rows), on='cell', how='left')
has_wx = grid[FEATURES].notna().all(axis=1)
print(f'Features for {has_wx.sum()} of {len(grid)} cells')

## 6. Predict risk in every cell

Besides the prediction, each cell is checked for **extrapolation**: if any feature lies outside the range seen in the training trials,
the model is predicting conditions it never learned from, and the cell is flagged. These cells are hatched on the map.

In [ ]:
X = grid.loc[has_wx, FEATURES]
grid.loc[has_wx, 'p_outbreak'] = outbreak_model.predict_proba(X)[:, 1]
grid.loc[has_wx, 'severity_pred'] = np.clip(severity_model.predict(X), 1, 5)
LEVELS = ['Green', 'Yellow', 'Orange', 'Red']
grid['alert'] = grid.p_outbreak.apply(lambda p: LEVELS[min(int(p // 0.25), 3)] if pd.notna(p) else None)

lo, hi = train[FEATURES].min(), train[FEATURES].max()
out_of_range = ((grid[FEATURES] < lo) | (grid[FEATURES] > hi)) & grid[FEATURES].notna()
grid['n_outside'] = out_of_range.sum(axis=1)
grid['extrapolated'] = grid['n_outside'] > 0
print('Alert levels (share of cells):', grid.alert.value_counts(normalize=True).reindex(LEVELS).round(2).fillna(0).to_dict())
print(f'Cells with weather outside the training range: {grid.extrapolated[has_wx].mean():.0%}')
print('Features most often outside the range:', out_of_range.sum().sort_values(ascending=False).head(4).to_dict())

## 7. Maps

In [ ]:
def to_array(col):
    A = np.full((len(lats), len(lons)), np.nan)
    A[grid.row, grid.col] = grid[col].astype(float)
    return A

def outline(ax):
    for g, col in [(shapes[c], 'k') for c in COUNTRIES] + [(lakes, '#2e86c1')]:
        polys = g.geoms if g.geom_type == 'MultiPolygon' else [g]
        for p in polys:
            x, y = p.exterior.xy
            ax.fill(x, y, color='#d6eaf8', zorder=1) if col != 'k' else None
            ax.plot(x, y, color=col, lw=0.7, zorder=2)

extent = [lons[0] - GRID_DEG / 2, lons[-1] + GRID_DEG / 2, lats[-1] - GRID_DEG / 2, lats[0] + GRID_DEG / 2]
season = f'{MAP_SEASON}/{str(MAP_SEASON + 1)[-2:]}'
fig, ax = plt.subplots(1, 3, figsize=(19, 6.2))

im = ax[0].imshow(to_array('p_outbreak'), extent=extent, cmap='YlOrRd', vmin=0, vmax=1, interpolation='nearest')
plt.colorbar(im, ax=ax[0], shrink=0.75, label='Outbreak probability')
ax[0].set_title(f'Outbreak probability, {season}')

alert_num = grid.alert.map({l: i for i, l in enumerate(LEVELS)})
A = np.full((len(lats), len(lons)), np.nan); A[grid.row, grid.col] = alert_num
cmap = ListedColormap(['#27ae60', '#f1c40f', '#e67e22', '#c0392b'])
ax[1].imshow(A, extent=extent, cmap=cmap, norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4), interpolation='nearest')
E = np.full((len(lats), len(lons)), np.nan); E[grid.row, grid.col] = grid.extrapolated.astype(float)
ax[1].contourf(lons, lats,
               np.where(E == 1, 1, np.nan), levels=[0.5, 1.5], colors='none', hatches=['///'])
ax[1].set_title(f'Alert level, {season} (hatched = outside training range)')
from matplotlib.patches import Patch
ax[1].legend(handles=[Patch(color=c, label=l) for c, l in zip(cmap.colors, LEVELS)], loc='lower left', fontsize=8)

im = ax[2].imshow(to_array('severity_pred'), extent=extent, cmap='viridis', vmin=1, vmax=3, interpolation='nearest')
plt.colorbar(im, ax=ax[2], shrink=0.75, label='Predicted mean severity (1–5)')
ax[2].set_title(f'Predicted severity, {season}')

for a in ax:
    outline(a); a.set_xlabel('Longitude'); a.set_ylabel('Latitude'); a.set_xlim(extent[:2]); a.set_ylim(extent[2:])
    if len(check):
        for ob, mk in [(1, '^'), (0, 'o')]:
            s = check[check.outbreak == ob]
            a.scatter(s.lon, s.lat, marker=mk, s=45, facecolor='white' if ob == 0 else 'black', edgecolor='k',
                      label='trial: outbreak' if ob else 'trial: no outbreak')
ax[0].legend(loc='lower left', fontsize=8)
plt.tight_layout(); plt.savefig(f'map_v2_{MAP_SEASON}_rust_risk.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
O = np.full((len(lats), len(lons)), np.nan); O[grid.row, grid.col] = (grid.sowing - fixed).dt.days
im = ax[0].imshow(O, extent=extent, cmap='Blues', interpolation='nearest'); outline(ax[0]); ax[0].set_xlim(extent[:2]); ax[0].set_ylim(extent[2:])
plt.colorbar(im, ax=ax[0], shrink=0.75, label='Days after 21 December'); ax[0].set_title('Estimated sowing date')
im = ax[1].imshow(to_array('wet_hours'), extent=extent, cmap='GnBu', interpolation='nearest'); outline(ax[1]); ax[1].set_xlim(extent[:2]); ax[1].set_ylim(extent[2:])
plt.colorbar(im, ax=ax[1], shrink=0.75, label='Hours with RH ≥ 90%'); ax[1].set_title('Leaf-wetness hours in the 14 days before R4')
plt.tight_layout(); plt.show()

## 8. Does the map agree with the season's trials?

The trials of the map season were **not** used in training (when `LEAVE_OUT_MAP_SEASON = True`). Their observed outcome is compared with
the map cell they fall in. The trial prediction uses the trial's own sowing date; the map uses the estimated sowing date, so both are shown.

In [ ]:
if len(check):
    rc = ((lats[0] + GRID_DEG / 2 - check.lat) / GRID_DEG).astype(int), ((check.lon - (lons[0] - GRID_DEG / 2)) / GRID_DEG).astype(int)
    key = dict(zip(zip(grid.row, grid.col), grid.index))
    idx = [key.get((r, c)) for r, c in zip(*rc)]
    comp = check[['env', 'country', 'loc', 'sowing', 'severity', 'outbreak']].copy()
    comp['p_trial_weather'] = outbreak_model.predict_proba(check[FEATURES])[:, 1]
    comp['map_cell_p'] = [grid.loc[i, 'p_outbreak'] if i is not None else np.nan for i in idx]
    comp['map_cell_alert'] = [grid.loc[i, 'alert'] if i is not None else None for i in idx]
    comp['map_cell_sowing'] = [grid.loc[i, 'sowing'] if i is not None else pd.NaT for i in idx]
    display(comp.round(2))
    ok = comp.dropna(subset=['map_cell_p'])
    if ok.outbreak.nunique() == 2:
        print(f'Mean map probability: outbreak trials {ok[ok.outbreak == 1].map_cell_p.mean():.2f} '
              f'vs no-outbreak trials {ok[ok.outbreak == 0].map_cell_p.mean():.2f}')
    print(f'Outbreak trials in Orange/Red map cells: {((ok.outbreak == 1) & ok.map_cell_alert.isin(["Orange", "Red"])).sum()} '
          f'of {int(ok.outbreak.sum())}')
else:
    print('No assessed trials in this season to compare with.')

## 9. Save maps and data

In [ ]:
tf = from_origin(lons[0] - GRID_DEG / 2, lats[0] + GRID_DEG / 2, GRID_DEG, GRID_DEG)
saved = [f'map_v2_{MAP_SEASON}_rust_risk.png']
for col in ['p_outbreak', 'severity_pred']:
    fn = f'map_v2_{MAP_SEASON}_{col}.tif'
    with rasterio.open(fn, 'w', driver='GTiff', height=len(lats), width=len(lons), count=1, dtype='float32',
                       crs='EPSG:4326', transform=tf, nodata=np.nan) as dst:
        dst.write(to_array(col).astype('float32'), 1)
    saved.append(fn)
cols = ['cell', 'lat', 'lon', 'country', 'onset', 'sowing', 'R4_date', 'p_outbreak', 'alert', 'severity_pred',
        'extrapolated', 'n_outside'] + FEATURES
grid[cols].to_csv(f'map_v2_{MAP_SEASON}_cells.csv', index=False); saved.append(f'map_v2_{MAP_SEASON}_cells.csv')
if len(check):
    comp.to_csv(f'map_v2_{MAP_SEASON}_trial_check.csv', index=False); saved.append(f'map_v2_{MAP_SEASON}_trial_check.csv')
print('Saved:', *saved, sep='\n  ')
try:
    from google.colab import files
    for f in saved: files.download(f)
except ImportError:
    pass

## Notes

* **Grid size:** 0.2° (~22 km) keeps the first map fast. For a finer map set `GRID_DEG = 0.1`; that needs about 4x more weather downloads and may hit the free
  hourly limit — the cache lets you re-run later and continue where it stopped.
* **Cropland:** every cell is shown, including forest, parks and lakes shores where no soybean grows. A cropland mask is a planned refinement.
* **Timing:** R4 is placed 69 days after sowing everywhere; cooler highlands develop more slowly, which is not yet modelled.
* **Other seasons:** change `MAP_SEASON` (2017–2024) and re-run. IFS weather is available from 2017; 2023/24 is the default because it contrasts with 2020/21.